In [4]:
# ============================================================================
# BLOCK 7: ENGAGEMENT METRICS - POLISH
# Input: 06_umap_data_pl.pkl
# Output: 07_engagement_data_pl.pkl + CSV exports
# Runtime: ~3 minutes
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 11:58:15
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [5]:
# CELL 1: Load previous checkpoint
print('='*70)
print('BLOCK 7: ENGAGEMENT METRICS - POLISH')
print('='*70)

if check_checkpoint_exists('pl', '07_engagement_data'):
    df_pl = load_checkpoint('pl', '07_engagement_data')
    print('✓ Loading from engagement checkpoint')
else:
    df_pl = load_checkpoint('pl', '06_umap_data')
    if df_pl is None:
        raise FileNotFoundError('Run 05_umap_visualization_pl.ipynb first')

BLOCK 7: ENGAGEMENT METRICS - POLISH
✓ Loading checkpoint: 06_umap_data_pl.pkl


In [6]:
# CELL 2: Engagement by sentiment
print('\n--- ENGAGEMENT BY SENTIMENT ---\n')

engagement_sentiment = df_pl.groupby('sentiment').agg({
    'comment_likes': ['mean', 'median', 'sum', 'count'],
    'replies': ['mean', 'median', 'sum']
}).round(2)

print('Likes by Sentiment:')
display(engagement_sentiment['comment_likes'])

print('\nReplies by Sentiment:')
display(engagement_sentiment['replies'])


--- ENGAGEMENT BY SENTIMENT ---

Likes by Sentiment:


,mean,median,sum,count
sentiment,,,,
Negative,34.10,1.0,149278.0,4378
Neutral,19.84,1.0,13389.0,675
Positive,36.97,1.0,63213.0,1710



Replies by Sentiment:


,mean,median,sum
sentiment,,,
Negative,1.24,0.0,5425.0
Neutral,0.84,0.0,570.0
Positive,0.98,0.0,1668.0


In [7]:
# CELL 3: Engagement by topic
print('\n--- ENGAGEMENT BY TOPIC ---\n')

engagement_topic = df_pl.groupby('topic').agg({
    'comment_likes': ['mean', 'median', 'sum'],
    'replies': ['mean', 'median', 'sum'],
    'clean_text': 'count'
}).round(2)
engagement_topic.columns = ['likes_mean', 'likes_median', 'likes_sum', 
                            'replies_mean', 'replies_median', 'replies_sum',
                            'comment_count']
engagement_topic = engagement_topic.sort_values('comment_count', ascending=False)

print('Top Topics by Engagement:')
display(engagement_topic.head(8))


--- ENGAGEMENT BY TOPIC ---

Top Topics by Engagement:


,likes_mean,likes_median,likes_sum,replies_mean,replies_median,replies_sum,comment_count
topic,,,,,,,
0.0,43.50,2.0,125877.0,1.47,0.0,4256.0,2894
-1.0,28.17,1.0,72201.0,0.98,0.0,2516.0,2563
1.0,15.89,1.0,8979.0,0.56,0.0,317.0,565
2.0,13.58,0.0,4143.0,0.66,0.0,202.0,305
3.0,20.13,4.0,4449.0,0.70,0.0,155.0,221
4.0,44.93,1.0,6694.0,1.06,0.0,158.0,149
5.0,71.53,2.5,2575.0,1.00,0.0,36.0,36
6.0,32.07,1.0,962.0,0.77,0.0,23.0,30


In [8]:
# CELL 4: Cross-tab (Sentiment × Topic)
print('\n--- SENTIMENT × TOPIC CROSS-TAB ---\n')

cross_tab = pd.crosstab(df_pl['topic'], df_pl['sentiment'], 
                        values=df_pl['comment_likes'], aggfunc='mean').round(2)
display(cross_tab)


--- SENTIMENT × TOPIC CROSS-TAB ---



sentiment,Negative,Neutral,Positive
topic,,,
-1.0,25.48,21.00,38.03
0.0,46.06,22.77,45.62
1.0,12.98,16.07,18.48
2.0,8.28,5.38,31.79
3.0,15.61,9.05,41.74
4.0,69.28,2.52,22.90
5.0,85.83,16.33,9.25
6.0,45.17,14.00,11.62


In [9]:
# CELL 5: Engagement heatmap
print('\nGenerating engagement heatmap...')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(cross_tab, annot=True, fmt='.0f', cmap='YlOrRd', 
            cbar_kws={'label': 'Average Likes'})
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Topic', fontsize=12)
ax.set_title('Average Likes by Topic × Sentiment - Polish', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'pl_engagement_heatmap.png', 'pl', 'visualizations')


Generating engagement heatmap...
✓ Saved: outputs\pl\visualizations\pl_engagement_heatmap.png


In [10]:
# CELL 6: Export engagement summary
engagement_summary = {
    'by_sentiment': engagement_sentiment.to_dict(),
    'by_topic': engagement_topic.to_dict(),
    'cross_tab': cross_tab.to_dict()
}

engagement_df = df_pl.groupby('sentiment').agg({
    'comment_likes': 'mean',
    'replies': 'mean',
    'clean_text': 'count'
}).reset_index()
engagement_df.columns = ['sentiment', 'avg_likes', 'avg_replies', 'comment_count']
engagement_df.to_csv(OUTPUT_DIR / 'polish' / 'pl_engagement_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'polish' / 'pl_engagement_summary.csv'}")


✓ Saved: outputs\polish\pl_engagement_summary.csv


In [11]:
# CELL 7: Export final dataset
final_cols = ['video_id', 'author_name', 'comment_text', 'clean_text', 
              'comment_date', 'comment_likes', 'replies',
              'sentiment', 'topic', 'topic_probability', 'umap_x', 'umap_y']
final_cols = [c for c in final_cols if c in df_pl.columns]

df_pl[final_cols].to_csv(OUTPUT_DIR / 'polish' / 'pl_final_dataset.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'polish' / 'pl_final_dataset.csv'}")

# Save checkpoint
save_checkpoint(df_pl, 'pl', '07_engagement_data')
update_pipeline_status('pl', 7, 'completed')

print('\n' + '='*70)
print('✓ BLOCK 7 COMPLETE - POLISH PIPELINE FINISHED')
print('='*70)
print(f'\nAll Polish outputs saved to: {OUTPUT_DIR / "polish"}')
print('='*70)


✓ Saved: outputs\polish\pl_final_dataset.csv
✓ Checkpoint saved: 07_engagement_data_pl.pkl
✓ Pipeline status updated: pl - Block 7 - completed

✓ BLOCK 7 COMPLETE - POLISH PIPELINE FINISHED

All Polish outputs saved to: outputs\polish
